In [ ]:
# Drive'ı bağla ve Sena'nın fonksiyonlarını kullanmak için yolu ekle
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.append('/content/drive/MyDrive/CodeReviewBot/utils/')

# Gerekli araçları yükle
!pip install datasets transformers torch

Mounted at /content/drive


In [ ]:
from huggingface_hub import notebook_login

# Bu komut interaktif bir giriş paneli açar
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from datasets import load_dataset

ds = load_dataset('code_search_net', 'python')
print(ds)
print('İlk örnek:')
print(ds['train'][0])


DatasetDict({
    train: Dataset({
        features: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url'],
        num_rows: 412178
    })
    test: Dataset({
        features: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url'],
        num_rows: 22176
    })
    validation: Dataset({
        features: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url'],
        num_rows: 23107
    })
})
İlk örnek:
{'repository_name': 'mjirik/imcut', 'func_path_in_repository': 'imcut/pycut.py', 'func_n

In [ ]:
save_path = '/content/drive/MyDrive/CodeReviewBot/data/codesearchnet/'
ds.save_to_disk(save_path)
print('CodeSearchNet kaydedildi!')


Saving the dataset (0/4 shards):   0%|          | 0/412178 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/22176 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/23107 [00:00<?, ? examples/s]

CodeSearchNet kaydedildi!


In [ ]:
import os
os.makedirs('/content/drive/MyDrive/CodeReviewBot/data/codereviewer_raw/', exist_ok=True)

base = '/content/drive/MyDrive/CodeReviewBot/data/codereviewer_raw/'
base_url = 'https://zenodo.org/records/6900648/files/'

# 3 ayrı zip'i indir
!wget -P {base} {base_url}Comment_Generation.zip
!wget -P {base} {base_url}Code_Refinement.zip
!wget -P {base} {base_url}Diff_Quality_Estimation.zip

# Hepsini aç
!unzip {base}Comment_Generation.zip -d {base}
!unzip {base}Code_Refinement.zip -d {base}
!unzip {base}Diff_Quality_Estimation.zip -d {base}

print('Tüm dosyalar indirildi!')

--2026-04-27 18:42:52--  https://zenodo.org/records/6900648/files/Comment_Generation.zip
Resolving zenodo.org (zenodo.org)... 188.185.43.153, 137.138.153.219, 188.184.98.114, ...
Connecting to zenodo.org (zenodo.org)|188.185.43.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 846602315 (807M) [application/octet-stream]
Saving to: ‘/content/drive/MyDrive/CodeReviewBot/data/codereviewer_raw/Comment_Generation.zip’

Comment_Generation. 100%[===================>] 807.38M  29.4MB/s    in 29s     

2026-04-27 18:43:21 (28.2 MB/s) - ‘/content/drive/MyDrive/CodeReviewBot/data/codereviewer_raw/Comment_Generation.zip’ saved [846602315/846602315]

--2026-04-27 18:43:21--  https://zenodo.org/records/6900648/files/Code_Refinement.zip
Resolving zenodo.org (zenodo.org)... 188.185.43.153, 137.138.153.219, 188.184.98.114, ...
Connecting to zenodo.org (zenodo.org)|188.185.43.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1168582011 (1.1G) [appl

In [ ]:
import os, json

path = '/content/drive/MyDrive/CodeReviewBot/data/codereviewer_raw/'

# Tüm jsonl dosyalarını listele
for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith('.jsonl'):
            full = os.path.join(root, file)
            size = os.path.getsize(full)
            print(f'{full}  →  {size} byte')

/content/drive/MyDrive/CodeReviewBot/data/codereviewer_raw/train.jsonl  →  0 byte
/content/drive/MyDrive/CodeReviewBot/data/codereviewer_raw/Comment_Generation/msg-train.jsonl  →  3621670365 byte
/content/drive/MyDrive/CodeReviewBot/data/codereviewer_raw/Comment_Generation/msg-valid.jsonl  →  273738946 byte
/content/drive/MyDrive/CodeReviewBot/data/codereviewer_raw/Comment_Generation/msg-test.jsonl  →  275294476 byte
/content/drive/MyDrive/CodeReviewBot/data/codereviewer_raw/Code_Refinement/ref-test.jsonl  →  394922734 byte
/content/drive/MyDrive/CodeReviewBot/data/codereviewer_raw/Code_Refinement/ref-train.jsonl  →  5320097797 byte
/content/drive/MyDrive/CodeReviewBot/data/codereviewer_raw/Code_Refinement/ref-valid.jsonl  →  389557763 byte
/content/drive/MyDrive/CodeReviewBot/data/codereviewer_raw/Diff_Quality_Estimation/cls-test.jsonl  →  1466531552 byte
/content/drive/MyDrive/CodeReviewBot/data/codereviewer_raw/Diff_Quality_Estimation/cls-train-chunk-0.jsonl  →  2971621227 byte
/con

In [ ]:
import json, os

train_path = '/content/drive/MyDrive/CodeReviewBot/data/codereviewer_raw/Comment_Generation/msg-train.jsonl'
clean_dir = '/content/drive/MyDrive/CodeReviewBot/data/codereviewer_clean/'
os.makedirs(clean_dir, exist_ok=True)

# Önce ilk 2 örneği göster - alan adlarını görelim
print('=== İLK 2 ÖRNEK ===')
with open(train_path, 'r') as f:
    for i, line in enumerate(f):
        item = json.loads(line)
        print(f'Örnek {i+1} alanlar:', list(item.keys()))
        print(f'Örnek {i+1}:', item)
        print()
        if i == 1:
            break

# Temizle - dosya çok büyük olduğu için ilk 50.000 örneği al
print('=== TEMİZLEME BAŞLIYOR ===')
clean_data = []
with open(train_path, 'r') as f:
    for i, line in enumerate(f):
        if i >= 50000:  # Colab için yeterli, tamamı 3.6GB
            break
        item = json.loads(line)
        # Boş olmayanları al
        old = item.get('old_hunk') or item.get('old_file') or item.get('patch') or ''
        comment = item.get('comment') or item.get('msg') or ''
        if old.strip() and comment.strip():
            clean_data.append(item)

with open(clean_dir + 'train_clean.jsonl', 'w') as f:
    for item in clean_data:
        f.write(json.dumps(item) + '\n')

print(f'Temiz veri: {len(clean_data)} örnek kaydedildi!')

=== İLK 2 ÖRNEK ===
Örnek 1 alanlar: ['oldf', 'patch', 'msg', 'id', 'y']
Örnek 1: {'oldf': '//    |  /           |\n//    \' /   __| _` | __|  _ \\   __|\n//    . \\  |   (   | |   (   |\\__ `\n//   _|\\_\\_|  \\__,_|\\__|\\___/ ____/\n//                   Multi-Physics\n//\n//  License:\t\t BSD License\n//\t\t\t\t\t Kratos default license: kratos/license.txt\n//\n//  Main authors:    Vicente Mataix Ferrandiz\n//\n\n// System includes\n#include <limits>\n\n// External includes\n\n\n// Project includes\n#include "testing/testing.h"\n\n// Utility includes\n#include "utilities/math_utils.h"\n\nnamespace Kratos \n{\n    namespace Testing \n    {\n        /// Tests\n       \n        /** Checks if the area of the triangle is calculated correctly using Heron equation.\n         * Checks if the area of the triangle is calculated correctly using Heron equation.\n         */\n        \n        KRATOS_TEST_CASE_IN_SUITE(MathUtilsHeronTest, KratosCoreMathUtilsFastSuite) \n        {\n            co

In [ ]:
import json, os

train_path = '/content/drive/MyDrive/CodeReviewBot/data/codereviewer_raw/Comment_Generation/msg-train.jsonl'
clean_dir = '/content/drive/MyDrive/CodeReviewBot/data/codereviewer_clean/'
os.makedirs(clean_dir, exist_ok=True)

clean_data = []
skipped = 0

with open(train_path, 'r') as f:
    for i, line in enumerate(f):
        if i >= 50000:
            break
        item = json.loads(line)
        oldf = item.get('oldf', '').strip()
        patch = item.get('patch', '').strip()
        msg = item.get('msg', '').strip()

        # Boş olmayanları al, y=1 olanları tercih et (yorum gerektiren değişiklikler)
        if oldf and patch and msg:
            clean_data.append({
                'input': patch,   # CodeT5+'a giriş olarak diff/patch kullanacağız
                'oldf': oldf,
                'msg': msg,
                'y': item.get('y', 1)
            })
        else:
            skipped += 1

with open(clean_dir + 'train_clean.jsonl', 'w') as f:
    for item in clean_data:
        f.write(json.dumps(item) + '\n')

print(f'Temiz veri: {len(clean_data)} örnek kaydedildi!')
print(f'Atlanan (boş alan): {skipped}')
print(f'İlk örnek input (patch): {clean_data[0]["input"][:200]}')
print(f'İlk örnek msg: {clean_data[0]["msg"]}')

Temiz veri: 48874 örnek kaydedildi!
Atlanan (boş alan): 1126
İlk örnek input (patch): @@ -595,8 +595,10 @@ namespace Kratos
             array_1d<double, 3> b = ZeroVector(3);
             b[0] = 1.0;
 
-            const array_1d<double, 3>  c = MathUtils<double>::CrossProduct(a, b);

İlk örnek msg: I assumed that for CrossProduct the values were inverted as well... Is that right?


In [ ]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive')

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive


In [ ]:
import os
for root, dirs, files in os.walk('/content/drive/MyDrive/'):
    if 'CodeReviewBot' in dirs:
        print('Bulundu:', root)
        break

Bulundu: /content/drive/MyDrive/


In [ ]:
!pip uninstall transformers sentence-transformers -y
!pip install transformers==4.44.0 -q

Found existing installation: transformers 4.44.0
Uninstalling transformers-4.44.0:
  Successfully uninstalled transformers-4.44.0


In [ ]:
!pip install transformers datasets torch sacrebleu -q

In [ ]:
# HÜCRE 2 — Model ve tokenizer yükle
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "Salesforce/codet5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Model yüklendi! Device: {device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/892M [00:00<?, ?B/s]

Model yüklendi! Device: cuda


In [ ]:
# HÜCRE 3 — Veriyi yükle ve Dataset'e çevir
import json
from torch.utils.data import Dataset, DataLoader
import torch

clean_path = '/content/drive/MyDrive/CodeReviewBot/data/codereviewer_clean/train_clean.jsonl'

data = []
with open(clean_path, 'r') as f:
    for line in f:
        data.append(json.loads(line))

print(f'Toplam örnek: {len(data)}')
print(f'İlk örnek: {data[0]}')


Toplam örnek: 48874
İlk örnek: {'input': '@@ -595,8 +595,10 @@ namespace Kratos\n             array_1d<double, 3> b = ZeroVector(3);\n             b[0] = 1.0;\n \n-            const array_1d<double, 3>  c = MathUtils<double>::CrossProduct(a, b);\n-            const array_1d<double, 3>  d = MathUtils<double>::UnitCrossProduct(a, b);\n+            array_1d<double, 3>  c, d;\n+\n+            MathUtils<double>::CrossProduct(c, b, a);\n+            MathUtils<double>::UnitCrossProduct(d, b, a);\n             \n             KRATOS_CHECK_EQUAL(c[2], 2.0);\n             KRATOS_CHECK_EQUAL(d[2], 1.0);', 'oldf': '//    |  /           |\n//    \' /   __| _` | __|  _ \\   __|\n//    . \\  |   (   | |   (   |\\__ `\n//   _|\\_\\_|  \\__,_|\\__|\\___/ ____/\n//                   Multi-Physics\n//\n//  License:\t\t BSD License\n//\t\t\t\t\t Kratos default license: kratos/license.txt\n//\n//  Main authors:    Vicente Mataix Ferrandiz\n//\n\n// System includes\n#include <limits>\n\n// External includes\

In [ ]:

# HÜCRE 4 — Tokenize et
class CodeReviewDataset(Dataset):
    def __init__(self, data, tokenizer, max_input=512, max_target=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_input = max_input
        self.max_target = max_target

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        input_enc = self.tokenizer(
            item['input'],
            max_length=self.max_input,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        target_enc = self.tokenizer(
            item['msg'],
            max_length=self.max_target,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        labels = target_enc['input_ids'].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100  # loss hesabında padding'i yoksay

        return {
            'input_ids': input_enc['input_ids'].squeeze(),
            'attention_mask': input_enc['attention_mask'].squeeze(),
            'labels': labels
        }

dataset = CodeReviewDataset(data, tokenizer)
print(f'Dataset hazır, {len(dataset)} örnek')

Dataset hazır, 48874 örnek


In [ ]:
# HÜCRE 5 — Eğitim
from transformers import Trainer, TrainingArguments
import os

checkpoint_dir = '/content/drive/MyDrive/CodeReviewBot/models/codet5_checkpoint/'
os.makedirs(checkpoint_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir=checkpoint_dir,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    warmup_steps=200,
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

print("Eğitim başlıyor...")
trainer.train()
print("Eğitim tamamlandı!")



Eğitim başlıyor...


Step,Training Loss
100,6.387200
200,4.002300
300,3.828700
400,3.895600
500,3.820800
600,3.885300
700,3.843900
800,3.696600
900,3.806400
1000,3.795000


Eğitim tamamlandı!


In [ ]:
import os
checkpoint_dir = '/content/drive/MyDrive/CodeReviewBot/models/codet5_checkpoint/'
print(os.listdir(checkpoint_dir))

['checkpoint-36500', 'checkpoint-36657']


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

checkpoint_dir = '/content/drive/MyDrive/CodeReviewBot/models/codet5_checkpoint/'
last_checkpoint = checkpoint_dir + 'checkpoint-36657'

# Her ikisini de checkpoint'ten yükle
model = AutoModelForSeq2SeqLM.from_pretrained(last_checkpoint)
tokenizer = AutoTokenizer.from_pretrained(last_checkpoint)

# Ana klasöre kaydet
model.save_pretrained(checkpoint_dir)
tokenizer.save_pretrained(checkpoint_dir)
print("Model kaydedildi!")

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model kaydedildi!


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Checkpoint'i yükle
checkpoint = '/content/drive/MyDrive/CodeReviewBot/models/codet5_checkpoint/checkpoint-36657'
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

# Test et
test = "@@ -1 +1 @@ -x = 1 +x = 2"
inputs = tokenizer(test, return_tensors="pt", max_length=512, truncation=True)
with torch.no_grad():
    out = model.generate(inputs["input_ids"], max_length=128)
print("Test:", repr(tokenizer.decode(out[0], skip_special_tokens=True)))

# Çalışıyorsa ana klasöre kaydet
model.save_pretrained('/content/drive/MyDrive/CodeReviewBot/models/codet5_checkpoint/')
tokenizer.save_pretrained('/content/drive/MyDrive/CodeReviewBot/models/codet5_checkpoint/')
print("Kaydedildi!")

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Test: ''


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Kaydedildi!


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Trainer, TrainingArguments
import json, torch
from torch.utils.data import Dataset

# Modeli yükle
checkpoint = '/content/drive/MyDrive/CodeReviewBot/models/codet5_checkpoint/checkpoint-36657'
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

# Veriyi yükle
data = []
with open('/content/drive/MyDrive/CodeReviewBot/data/codereviewer_clean/train_clean.jsonl') as f:
    for line in f:
        data.append(json.loads(line))

class CodeReviewDataset(Dataset):
    def __init__(self, data, tokenizer):   # __ çift alt çizgi __
        self.data = data
        self.tokenizer = tokenizer
    def __len__(self):                     # __ çift alt çizgi __
        return len(self.data)
    def __getitem__(self, idx):            # __ çift alt çizgi __
        item = self.data[idx]
        inp = self.tokenizer(item['input'], max_length=512, truncation=True, padding='max_length', return_tensors='pt')
        tgt = self.tokenizer(item['msg'], max_length=128, truncation=True, padding='max_length', return_tensors='pt')
        labels = tgt['input_ids'].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {'input_ids': inp['input_ids'].squeeze(), 'attention_mask': inp['attention_mask'].squeeze(), 'labels': labels}

dataset = CodeReviewDataset(data, tokenizer)

training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/CodeReviewBot/models/codet5_checkpoint/',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    save_steps=500,
    save_total_limit=2,
    fp16=True,
    report_to="none"
)

trainer = Trainer(model=model, args=training_args, train_dataset=dataset)
print("Eğitim başlıyor...")
trainer.train()

model.save_pretrained('/content/drive/MyDrive/CodeReviewBot/models/codet5_checkpoint/')
tokenizer.save_pretrained('/content/drive/MyDrive/CodeReviewBot/models/codet5_checkpoint/')
print("Bitti!")